# Modelling a glacier with IGM — Session 3: experiment deepening

In Session 2 you went through the **basic workflow** (data, calibration of the ice flow and of the melt, past, future, ELA retrieval) with the Rhone Glacier, then with your own glacier. Session 3 **deepens** some experiments on your glacier, by bringing new data and by customising IGM:

1. **The Little Ice Age extent** (**everyone**): compare the modelled glacier in 1850 with the mapped 1850 extent, and refine your calibration of the past
2. **Extension A — the response time**: write / read your own IGM module (oscillating ELA)
3. **Extension B — the moraines**: track debris with particles (if relevant for your glacier)
4. **Figures for the report**

Part 1 (fine-tuning the calibration on the LIA extent) is **necessary for everyone**. Parts 2–3 are **additional topics**, to be taken afterwards **if time allows**.

Keep working in **your** project `~/work/my-glacier` (copied in Session 2): do **not** copy the template again. Just update the course material (new plotting helpers) and add this notebook to your project, in a terminal:

```bash
cd ~/work/glamod && git pull
cp ~/work/glamod/session3/tutorial-session3.ipynb ~/work/my-glacier/
```

then open `~/work/my-glacier/tutorial-session3.ipynb` (it reads the runs of `~/work/my-glacier/outputs/`).

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/params_past.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here

## 1. The Little Ice Age extent: a closer calibration of the past

**Why?** In Session 2 you calibrated the melt (`melt_enhancer`) mostly on today's volume, and checked the LIA maximum by eye. The LIA maximum (~1850) is the best-documented past state of Alpine glaciers: its **moraines** are still visible, and it was mapped on the first topographic maps. Comparing the modelled extent in 1850 with the mapped one is a strong test of the model, and helps to constrain the **past climate** (pre-1901 `delta_temp`) and the **melt**.

**Resources for the 1850 extent**:
- **Swiss Glacier Inventory 1850** (SGI 1850, [GLAMOS](https://www.glamos.ch/en/downloads#inventories); Maisch et al., 2000): outlines of all Swiss glaciers in 1850, from historical maps and moraines. Downloaded automatically below (`data/sgi1850/`).
- **Historical maps** on [map.geo.admin.ch](https://map.geo.admin.ch) (*Journey through time*): Dufour map (1845–1865) and Siegfried map (1870–1926), close to the LIA maximum.
- **Moraines** on the relief (*swissALTI3D* hillshade) and aerial images (*SWISSIMAGE*) of map.geo.admin.ch: the LIA moraines are the youngest, fresh, unvegetated ridges in front of the glacier.
- **Length changes** since ~1880 (GLAMOS factsheet of your glacier, e.g. [Rhone Glacier](https://www.glamos.ch/en/factsheet#/B43-03)): the retreat of the front.

▶ First compare your calibrated past run (Session 2) with the 1850 outline:

In [ ]:
run = "outputs/params_past"                         # <- your calibrated past run (Session 2)
compare_lia_extent(run, INPUT, year=1850)
compare_runs([run])

**Then refine the calibration of the past**: copy `experiment/params_past.yaml` (e.g. `params_past_lia.yaml`, with the run folder `outputs/params_past_lia` at the end of the file), and adjust one parameter at a time:
- the glacier is **too small / too large in 1850** ➡️ colder / warmer pre-1901 `delta_temp` (rows 1700 and 1850 of `clim_trend_array`)
- the glacier is **too small / too large today** ➡️ more / less melt: `melt_enhancer`
- the front is too thin and advances too slowly / too fast ➡️ check the ice flow again (`arrhenius`, `tau_ref`, Session 2, section 2)

```bash
igm_run +experiment=params_past_lia
```

Aim for a run that reaches the 1850 outline **and** today's volume; no perfect fit is possible, **discuss** the remaining mismatch in your report. If your calibration changes, redo the future scenarios (`params_future.yaml`).

In [ ]:
run = "outputs/params_past_lia"
compare_lia_extent(run, INPUT, year=1850)
compare_runs(["outputs/params_past", run])

> ✅ **Calibration of the past, refined** (double-click to edit): pre-1901 `delta_temp` = ...  `melt_enhancer` = ...  (`arrhenius` = ...  `tau_ref` = ...)

## 2. Extension A — Response time (if time allows): your own IGM module

**Why?** In Session 2 (ELA retrieval) you saw that the glacier does not react immediately to an ELA change. How long does it take? This **response time** $\tau$ tells how much of the present-day warming your glacier has not "seen" yet, and how much it will keep retreating even if the climate stabilises.

**The experiment**: the ELA oscillates like a sine wave (period $P$ = 200 yr, amplitude 200 m) and we look at how the glacier volume follows.

**Look at the module first**: this SMB is not part of IGM, it is a **user module** written in Python: `user/code/processes/smb_oscillating.py` (its parameters: `user/conf/processes/smb_oscillating.yaml`). Open it: like any IGM module, it has 3 functions, `initialize` (called once), `update` (at each time step: computes the ELA at time t, then the SMB from the surface elevation) and `finalize`. This is how you can extend IGM with your own processes.

### i) Find the response time

▶ **In the terminal** (400 model years, a few minutes):

```bash
igm_run +experiment=params_response_time
```

In [ ]:
plot_response_time("outputs/params_response_time")

The volume **lags behind** the ELA. If the glacier behaves like a first-order system ($dV/dt = (V_{eq} - V)/\tau$), the lag gives $\tau = \frac{P}{2\pi}\tan\left(\frac{2\pi\,\text{lag}}{P}\right)$ (printed above). This only works when the lag is clearly shorter than $P/4$: if not, the period is too short, use a longer one (ii).

### ii) Test other periods, compare with the theoretical value

Copy `experiment/params_response_time.yaml` to e.g. `experiment/params_response_800.yaml`, and set in it `period: 800`, `end: 1600` (always **two periods**) and the run folder `outputs/params_response_800`. Try short (50, 100 yr) and long (400, 800 yr) periods:

```bash
igm_run +experiment=params_response_800
```

How do the **lag** and the **amplitude** of the volume oscillation change with the period? Which periods give a reliable $\tau$?

In [ ]:
for P in [100, 800]:                                  # <- the periods you ran
    plot_response_time(f"outputs/params_response_{P}")

Compare with the **theoretical** (volume time scale) estimate $\tau \approx H / (-b_{\text{terminus}})$ (Lecture 1a; Jóhannesson et al., 1989), with $H$ a typical (here: maximum) ice thickness and $b_{\text{terminus}}$ the SMB at the front:

In [ ]:
ds = xr.open_dataset(INPUT)
thk, usurf = ds.thk.values, ds.usurf.values
H = thk.max()                                          # maximum ice thickness (m)
z_front = usurf[thk > 1].min()                         # elevation of the front (m)
ela = np.quantile(usurf[thk > 10], 0.4)                # mean ELA of the module (40 % quantile of the surface)
b_t = 0.009 * (z_front - ela)                          # SMB at the front (m/yr), ablation gradient 0.009
print(f"H = {H:.0f} m, front at {z_front:.0f} m, ELA {ela:.0f} m, b_terminus = {b_t:.1f} m/yr")
print(f"Theoretical response time: tau = H / (-b_terminus) = {H / -b_t:.0f} yr")

### iii) Relate the response time to the size of the glacier

- From $\tau \approx H / (-b_{\text{terminus}})$: how should the response time depend on the **thickness** of the glacier, and on the **elevation of its front**?
- Compare your $\tau$ and the volume of your glacier with **other students' glaciers** (small cirque glaciers vs large valley glaciers): does the response time grow with the volume? Why?

### iv) Conclusion: what does the glacier do with climate variations?

- With a period much **shorter** than $\tau$, how large is the volume oscillation? And with a period much **longer**?
- What happens then to year-to-year or decadal climate fluctuations? And to long-term trends?
- Is your glacier today in balance with the present climate? What does this mean for its evolution until 2100, even without further warming?
- Would a small glacier and a large one tell the same story about the climate?

> ✅ **Conclusion — response time** (double-click to edit): the response time of my glacier is about ... years (from the lag with $P$ = ... yr), vs ... years from $H/(-b_{\text{terminus}})$. Volume of my glacier: ... km³. Climate variations shorter than ... years are damped. Consequence for 2100: ...

## 3. Extension B — Moraines: reproduce the LIA moraines (if time allows)

**Why?** Rocks fallen on the glacier are buried, carried by the ice flow and released at the margins and the front, where they build **lateral and frontal moraines**. The LIA moraines are the youngest, fresh ridges in front of today's glaciers. Can the model reproduce **where** they are? IGM tracks the debris with **particles** moved by the 3D ice flow: when the glacier retreats after ~1850, the particles left on the ice-free terrain show where the modelled glacier dropped its debris.

▶ **In the terminal**: put your calibrations (ice flow, `melt_enhancer`, pre-1901 climate from part 1) in `experiment/params_past_moraines.yaml`, then (a few minutes, more than the past run)

```bash
igm_run +experiment=params_past_moraines
```

In [ ]:
run_p = "outputs/params_past_moraines"
plot_particles(run_p, year=1850)                    # debris in the glacier at the LIA maximum (colour: depth)
plot_moraines(run_p, INPUT, year=2020)              # debris left on ice-free terrain today = modelled LIA moraines

**Compare with the real moraines**: on [map.geo.admin.ch](https://map.geo.admin.ch), look at the aerial images (SWISSIMAGE) and the relief (swissALTI3D hillshade) around your glacier: where are the LIA lateral and frontal moraines (fresh, unvegetated ridges)? Are the modelled deposits at the same place, at the same height on the valley sides?

**Play with the controls** (`particles: seeding:` in `params_past_moraines.yaml`; copy the file and change its run folder for each try):
- `method`: `accumulation` (surface of the accumulation area, regularly) or `all` (the whole ice column, once at the start)
- `frequency` (years between two seedings) and `density` (fraction of grid cells seeded): how much debris
- your calibration of the past (part 1): a glacier too large / too small in 1850 puts its moraines too far / too close

**Discuss the limits**: real debris comes mostly from the **rock walls** above the glacier (rockfall), not from the whole accumulation area; the moraine ridges are built by processes not in the model (dumping, pushing, reworking).

## 4. Figures for the report

The helper functions in `utils/helper.py` are meant for quick looks. For the report, make your own figures: copy the code of a helper into a cell below and adapt it (years shown, titles, units, colors, fonts), then save it into `figures/` **before** `plt.show()`:

```python
plt.savefig("figures/fig_volume.png", dpi=200, bbox_inches="tight")
```

In [ ]:
# your figures here